<a href="https://colab.research.google.com/github/Andrian0s/ML4NLP1-2026-Tutorial-Notebooks/blob/main/tutorials_notebooks_in_class_2026/W08_intro_to_hugging_face_transformers_datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Quick Introduction to Hugging Face's Transformers and Datasets Libraries

BERT is a stack of Transformer encoder blocks that was pretrained on a large amount of text. To use it for a specific task, we put a new classification head on top of it and fine-tune the model on labelled examples, for example to detect spam. In this tutorial, we do exactly this with the Hugging Face libraries, in four steps:

1. load a dataset with `datasets`;
2. turn the texts into token IDs with a tokenizer;
3. load a pretrained model with a new classification head;
4. fine-tune and evaluate it with the `Trainer`.

Exercise 4 follows the same steps for token classification (named entity recognition).

- Transformers documentation: https://huggingface.co/docs/transformers/index
- Fine-tuning guide: https://huggingface.co/docs/transformers/training
- Datasets documentation: https://huggingface.co/docs/datasets/index

In [1]:
%pip install -q transformers==5.16.1 datasets==5.1.0 accelerate==1.15.0

import torch
# Pick the best available device: NVIDIA GPU, then Apple Silicon GPU, then CPU
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(device)

Note: you may need to restart the kernel to use updated packages.


mps


## 1. Loading the dataset

We use the [SMS Spam Collection](https://huggingface.co/datasets/ucirvine/sms_spam): about 5,600 text messages, each labelled as *ham* (a normal message) or *spam*. The `datasets` library downloads it from the Hugging Face Hub.

In [2]:
import datasets

dataset = datasets.load_dataset("ucirvine/sms_spam")
dataset

DatasetDict({
    train: Dataset({
        features: ['sms', 'label'],
        num_rows: 5574
    })
})

This dataset has only a training split. With the `split` argument, we can load just a slice of it: here, the first 1,000 messages for training and the last 500 for testing. Slices like this also save time in Exercise 4.

In [3]:
train_set = datasets.load_dataset("ucirvine/sms_spam", split="train[:1000]")
test_set = datasets.load_dataset("ucirvine/sms_spam", split="train[-500:]")
print(train_set)
print(test_set)

Dataset({
    features: ['sms', 'label'],
    num_rows: 1000
})
Dataset({
    features: ['sms', 'label'],
    num_rows: 500
})


In [4]:
from collections import Counter

print(train_set.features["label"].names)
Counter(train_set["label"])

['ham', 'spam']


Counter({0: 848, 1: 152})

Only about 13% of the messages are spam. What accuracy would a model get that always predicts *ham*? Keep this in mind when we evaluate our model at the end.

In [5]:
train_set[0]

{'sms': 'Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...\n',
 'label': 0}

## 2. Tokenizer

The tokenizer converts text into the numbers that the model can process. It:

* splits the text into tokens, which are words or subwords (BERT uses WordPiece, GPT-2 uses byte-pair encoding);
* maps each token to its ID in the model's vocabulary;
* adds special tokens, such as `[CLS]` at the start and `[SEP]` at the end of a text for BERT.

Each model is trained with its own tokenizer. If you use a different one, the token IDs do not match what the model expects, and the predictions are useless. `AutoTokenizer` loads the right tokenizer for a model name, so the same code works for BERT, RoBERTa or XLM-R.

In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

Frequent words are single tokens, while rare words are split into several subwords. A subword that continues the previous token starts with `##`. Other tokenizers split the same word differently, because each tokenizer has its own vocabulary: the uncased BERT model, for example, gives `oct ##otho ##rp ##e`.

In [7]:
tokenizer.tokenize("The origins of the word octothorpe are shrouded in mystery.")

['The',
 'origins',
 'of',
 'the',
 'word',
 'o',
 '##ct',
 '##oth',
 '##or',
 '##pe',
 'are',
 's',
 '##hr',
 '##oud',
 '##ed',
 'in',
 'mystery',
 '.']

In [8]:
tokenizer(train_set[0]["sms"])

{'input_ids': [101, 3414, 1235, 179, 22497, 1403, 1553, 117, 4523, 119, 119, 11651, 8009, 2165, 1178, 1107, 15430, 1548, 183, 1632, 1362, 2495, 174, 171, 9435, 2105, 119, 119, 119, 140, 2042, 1175, 1400, 1821, 4474, 20049, 1204, 119, 119, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

For batches, all inputs must have the same length: shorter texts are padded with `[PAD]` tokens, and longer ones are truncated. To choose `max_length`, look at how long your texts are in tokens.

In [9]:
lengths = [len(tokenizer(text)["input_ids"]) for text in train_set["sms"]]
print("longest message:", max(lengths), "tokens")
print("messages longer than 64 tokens:", sum(length > 64 for length in lengths))

longest message: 106 tokens
messages longer than 64 tokens: 41


In [10]:
encoded = tokenizer(train_set[0]["sms"], padding="max_length", truncation=True, max_length=64)
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"]))
print(encoded["attention_mask"])

['[CLS]', 'Go', 'until', 'j', '##uron', '##g', 'point', ',', 'crazy', '.', '.', 'Ava', '##ila', '##ble', 'only', 'in', 'bug', '##is', 'n', 'great', 'world', 'la', 'e', 'b', '##uff', '##et', '.', '.', '.', 'C', '##ine', 'there', 'got', 'am', '##ore', 'wa', '##t', '.', '.', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


**input_ids** are the token IDs of the text. For BERT, they start with `[CLS]` (101) and end with `[SEP]` (102).

**attention_mask** is 1 for real tokens and 0 for padding, so that the model ignores the padding.

**token_type_ids** distinguish two texts that are given to the model together, for example a question and a paragraph: 0 for the first text and 1 for the second. BERT adds a learned segment vector for each of the two values to the token embeddings, in the same way as it adds position vectors. For a single text, they are all 0.

In [11]:
tokenizer("Is this spam?", "Yes, it is.")

{'input_ids': [101, 2181, 1142, 22620, 1306, 136, 102, 2160, 117, 1122, 1110, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

To tokenise a whole dataset, we use `Dataset.map` with `batched=True`, which processes many examples at once and returns a new `Dataset`. The `Trainer` expects the gold labels in a column called `labels`, and `set_format` returns the columns as PyTorch tensors.

In [12]:
def tokenize(batch):
    return tokenizer(batch["sms"], padding="max_length", truncation=True, max_length=64)

columns = ["input_ids", "token_type_ids", "attention_mask", "labels"]
encoded_train_set = train_set.map(tokenize, batched=True).rename_column("label", "labels")
encoded_test_set = test_set.map(tokenize, batched=True).rename_column("label", "labels")
encoded_train_set.set_format("torch", columns=columns)
encoded_test_set.set_format("torch", columns=columns)

for key, value in encoded_train_set[0].items():
    print(f"{key}: {value.shape}")

labels: torch.Size([])
input_ids: torch.Size([64])
token_type_ids: torch.Size([64])
attention_mask: torch.Size([64])


## 3. Model

A pretrained model consists of its architecture (the Transformer layers), its configuration (such as the number of layers and the hidden size) and its pretrained weights. For a task, we put a head on top of it: `AutoModelForSequenceClassification` adds a linear layer that maps the representation of the `[CLS]` token to one score per class. For Exercise 4, which is a token classification task, you use `AutoModelForTokenClassification` instead.

When the model is loaded, transformers prints a load report. The weights of the new classification head are *missing* from the checkpoint, so they are initialised randomly, while the heads that BERT used in pretraining are *unexpected* and are discarded: BERT was pretrained to predict masked words (masked language modelling) and to tell whether two sentences follow each other (next sentence prediction).

In [13]:
from transformers import AutoModelForSequenceClassification, set_seed

set_seed(42)  # the new head starts with random weights; a fixed seed makes reruns give the same result
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-cased", num_labels=2, id2label={0: "ham", 1: "spam"}, label2id={"ham": 0, "spam": 1})

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 26671.77it/s]


[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 4. Fine-tuning with the Trainer

In W05, you wrote the training loop yourself: batches, forward pass, loss, backward pass and optimiser step. The `Trainer` runs this loop for you and also takes care of evaluation, logging and moving the data to the GPU. `TrainingArguments` holds the hyperparameters, and `compute_metrics` tells the `Trainer` how to score the predictions on the evaluation set.

In [14]:
import numpy as np
from sklearn.metrics import f1_score
from transformers import Trainer, TrainingArguments


def compute_metrics(eval_pred):
    """Computes the F1 score of the spam class from the logits and the gold labels."""
    predictions = np.argmax(eval_pred.predictions, axis=-1)
    return {"f1": f1_score(eval_pred.label_ids, predictions)}


training_args = TrainingArguments(
    output_dir="results",
    num_train_epochs=1,
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_train_set,
    eval_dataset=encoded_test_set,
    processing_class=tokenizer,  # in older versions of transformers, this argument was called tokenizer
    compute_metrics=compute_metrics,
)

In [15]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,0.116619,0.043252,0.950820


TrainOutput(global_step=63, training_loss=0.11661948098076715, metrics={'train_runtime': 8.6787, 'train_samples_per_second': 115.224, 'train_steps_per_second': 7.259, 'total_flos': 32888881920000.0, 'train_loss': 0.11661948098076715, 'epoch': 1.0})

In [16]:
preds = trainer.predict(encoded_test_set)

In [17]:
print(preds.predictions[:2])             # raw scores (logits) for the first two messages
print(preds.predictions[:2].argmax(-1))  # the class with the highest score is the prediction
print(preds.label_ids[:2])               # gold labels
print(preds.metrics)

[[ 3.3539512 -3.0827796]
 [ 3.454035  -2.9640157]]
[0 0]
[0 0]
{'test_loss': 0.04325169324874878, 'test_f1': 0.9508196721311475, 'test_runtime': 0.947, 'test_samples_per_second': 527.962, 'test_steps_per_second': 8.447}


### Evaluation

Compare the accuracy with the F1 score of the spam class, and with the accuracy of a model that always predicts *ham*. In the confusion matrix, the rows are the gold labels and the columns are the predictions.

In [18]:
from sklearn.metrics import accuracy_score, confusion_matrix

predictions = preds.predictions.argmax(-1)
print("accuracy:", accuracy_score(preds.label_ids, predictions))
print("accuracy of always predicting ham:", accuracy_score(preds.label_ids, np.zeros_like(predictions)))
print("F1 of the spam class:", preds.metrics["test_f1"])
confusion_matrix(preds.label_ids, predictions)

accuracy: 0.988
accuracy of always predicting ham: 0.88
F1 of the spam class: 0.9508196721311475


array([[436,   4],
       [  2,  58]])

## 5. Pipelines

A `pipeline` wraps the tokenizer, the model and the post-processing into one call. We can use our fine-tuned classifier this way, and also the masked language modelling head that BERT was pretrained with: it predicts the word hidden behind `[MASK]` from both its left and its right context.

In [19]:
from transformers import pipeline

classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device=device)
classifier(["Congratulations! You have won a free ticket to Paris. Call 0800 123 456 now!",
            "Are we still meeting for lunch tomorrow?"])

[{'label': 'spam', 'score': 0.9983435869216919},
 {'label': 'ham', 'score': 0.9981062412261963}]

In [20]:
unmasker = pipeline("fill-mask", model="bert-base-cased", device=device)
for prediction in unmasker("The [MASK] ran to the emergency room to see her patient."):
    print(f"{prediction['token_str']:>10}  {prediction['score']:.3f}")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 15167.37it/s]


[transformers] BertForMaskedLM LOAD REPORT from: bert-base-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


     nurse  0.449
     woman  0.193
    doctor  0.074
      girl  0.053
    mother  0.036
